In [1]:
import pandas as pd 
from pathlib import Path
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression 
from sklearn.linear_model import Lasso
import warnings 
warnings.filterwarnings("ignore")
import pickle
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment01") 
mlflow.autolog()


"""
    mlflow.set_experiment is used to set the experiment name in MLflow.it works like a namespace for 
    organizing runs. Each experiment can have multiple runs, and you can track metrics, parameters, 
    and artifacts for each run within the experiment. If the experiment does not exist, 
    MLflow will create it automatically.
"""

cwd = Path.cwd().parent.parent
root_dir = Path( cwd / "01-intro")
def read_data(data_path:str) -> pd.DataFrame:
    data_path = Path(data_path)
    data_dir = root_dir / data_path
    print(data_dir.resolve())
    data01 = pd.read_parquet(data_dir)
    
    return data01

dataset = read_data("assignment/data/yellow_tripdata_2023-01.parquet")
dataset.head()

dataset['tpep_pickup_datetime'] = pd.to_datetime(dataset['tpep_pickup_datetime'])
dataset['tpep_dropoff_datetime'] = pd.to_datetime(dataset['tpep_dropoff_datetime'])

dataset['duration'] = (dataset['tpep_dropoff_datetime'] - dataset['tpep_pickup_datetime']).dt.total_seconds() / 60

dataset.head()

dataset['PULocationID'] = dataset['PULocationID'].astype(str)
dataset['DOLocationID'] = dataset['DOLocationID'].astype(str)

dict = dataset[['PULocationID','DOLocationID']].to_dict(orient='records')
dv = DictVectorizer()
X = dv.fit_transform(dict)


model = LinearRegression()
y = dataset['duration']
model.fit(X,y)
y_predicted = model.predict(X)
score = root_mean_squared_error(y,y_predicted)
print(score)

# did they just made the model determinsitcs

model = LinearRegression()
y = dataset['duration']
model.fit(X,y)
y_predicted = model.predict(X)
score = root_mean_squared_error(y,y_predicted)
print(score)

2025/06/04 19:49:10 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


/Users/alghali/Downloads/AI-Compeitions/mlops-zoomcamp/01-intro/assignment/data/yellow_tripdata_2023-01.parquet


2025/06/04 19:49:47 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'e5b66488533c4305a8ac1e4edae0ce05', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/06/04 19:51:07 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/06/04 19:53:06 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0751fa8687da466da29d30cebeb5bbc4', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


41.9964977084757


2025/06/04 19:53:52 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


41.9964977084757


In [2]:
# did they just made the model determinsitcs

model = LinearRegression()
y = dataset['duration']
model.fit(X,y)
y_predicted = model.predict(X)
score = root_mean_squared_error(y,y_predicted)
print(score)


with open("models/model.bin", "wb") as f_out:
    pickle.dump((dv, model), f_out)

2025/06/04 19:55:02 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'cef0adf07e324f799513b254a81016f9', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/06/04 19:55:45 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


41.9964977084757


In [3]:

with mlflow.start_run():
    
    mlflow.set_tag("developer", "Ahmed Alghali")
    mlflow.log_param("train_data_path", "/Users/alghali/Downloads/AI-Compeitions/mlops-zoomcamp/data")
    
    alpha=0.1
    mlflow.log_param("alpha",alpha)
    la = Lasso( random_state=42)
    la.fit(X, y)
    y_predicted = la.predict(X)
    score = root_mean_squared_error(y,y_predicted)
    mlflow.log_metric("RMSE",score)
    print(score)

2025/06/04 19:57:32 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/06/04 19:57:46 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 73d0e2a6f3db40c6bd4a5a4ee01ffaf3. Failed operations: [MlflowException("Changing param values is not allowed. Params were already logged=\'[{\'key\': \'alpha\', \'old_value\': \'0.1\', \'new_value\': \'1.0\'}]\' for run ID=\'73d0e2a6f3db40c6bd4a5a4ee01ffaf3\'.")]')]


42.55178089456983


In [4]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

# set the experiment id
mlflow.set_experiment(experiment_id="0")

mlflow.autolog()
db = load_diabetes()

X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)
rf.fit(X_train, y_train)

# Use the model to make predictions on the test dataset.
predictions = rf.predict(X_test)

2025/06/04 19:57:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/06/04 19:57:47 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd0f52523047f4ffaa8d370198a6ff09a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
